<a href="https://colab.research.google.com/github/Harry262000/ML_Assignment/blob/main/Harshal_Honde_ML_CpG_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Build CpG Detector

Here we have a simple problem, given a DNA sequence (of N, A, C, G, T), count the number of CpGs in the sequence (consecutive CGs).

We have defined a few helper functions / parameters for performing this task.

We need you to build a LSTM model and train it to complish this task in PyTorch.

A good solution will be a model that can be trained, with high confidence in correctness.

In [1]:
from typing import Sequence
from functools import partial
import random
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, random_split ,Dataset

In [2]:
# DO NOT CHANGE HERE
def set_seed(seed=13):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(13)

# Use this for getting x label
def rand_sequence(n_seqs: int, seq_len: int=128) -> Sequence[int]:
    for i in range(n_seqs):
        yield [random.randint(0, 4) for _ in range(seq_len)]

# Use this for getting y label
def count_cpgs(seq: str) -> int:
    cgs = 0
    for i in range(0, len(seq) - 1):
        dimer = seq[i:i+2]
        # note that seq is a string, not a list
        if dimer == "CG":
            cgs += 1
    return cgs

# Alphabet helpers
alphabet = 'NACGT'
dna2int = { a: i for a, i in zip(alphabet, range(5))}
int2dna = { i: a for a, i in zip(alphabet, range(5))}

intseq_to_dnaseq = partial(map, int2dna.get)
dnaseq_to_intseq = partial(map, dna2int.get)

In [3]:
sequences = list(rand_sequence(n_seqs=2048, seq_len=128))
print(sequences[:2])

[[2, 2, 1, 1, 1, 1, 1, 1, 0, 4, 1, 2, 0, 3, 1, 4, 0, 2, 1, 0, 2, 3, 3, 1, 2, 2, 1, 3, 4, 4, 3, 2, 3, 2, 0, 2, 4, 2, 3, 4, 4, 1, 3, 3, 4, 1, 2, 1, 1, 4, 2, 2, 2, 3, 2, 4, 2, 3, 1, 4, 3, 4, 1, 4, 1, 1, 2, 1, 0, 3, 3, 3, 0, 3, 0, 1, 1, 3, 3, 2, 1, 3, 2, 2, 1, 2, 4, 1, 4, 1, 3, 1, 4, 0, 4, 4, 0, 2, 4, 4, 2, 1, 2, 1, 1, 4, 0, 1, 1, 0, 3, 4, 4, 4, 0, 3, 2, 0, 4, 0, 1, 2, 0, 3, 2, 2, 4, 4], [0, 3, 4, 2, 0, 0, 2, 2, 1, 3, 4, 4, 4, 3, 1, 1, 2, 0, 3, 4, 1, 1, 2, 3, 1, 2, 3, 4, 1, 2, 1, 4, 2, 3, 1, 2, 3, 0, 2, 3, 3, 0, 0, 0, 4, 0, 2, 1, 3, 3, 4, 3, 2, 0, 2, 4, 3, 1, 1, 0, 0, 2, 1, 1, 2, 2, 4, 3, 2, 0, 2, 1, 0, 2, 3, 3, 4, 4, 3, 0, 2, 4, 2, 1, 0, 2, 1, 4, 1, 0, 1, 1, 0, 3, 2, 0, 2, 2, 1, 0, 1, 1, 4, 3, 1, 4, 3, 4, 0, 4, 0, 0, 4, 1, 4, 2, 4, 3, 2, 1, 4, 2, 2, 0, 2, 0, 0, 1]]


{N : 0, A : 1, C : 2, G : 3, T : 4}

In [4]:
dna_strings = [''.join(map(int2dna.get, seq))  for seq in sequences]
print(dna_strings[:2])

['CCAAAAAANTACNGATNCANCGGACCAGTTGCGCNCTCGTTAGGTACAATCCCGCTCGATGTATAACANGGGNGNAAGGCAGCCACTATAGATNTTNCTTCACAATNAANGTTTNGCNTNACNGCCTT', 'NGTCNNCCAGTTTGAACNGTAACGACGTACATCGACGNCGGNNNTNCAGGTGCNCTGAANNCAACCTGCNCANCGGTTGNCTCANCATANAANGCNCCANAATGATGTNTNNTATCTGCATCCNCNNA']


In [5]:
dna_strings_array = np.array(dna_strings)
print(dna_strings_array.shape)

(2048,)


In [6]:
# we prepared two datasets for training and evaluation
# training data scale we set to 2048
# we test on 512

def prepare_data(num_samples=100):
    # prepared the training and test data
    # you need to call rand_sequence and count_cpgs here to create the dataset
    # step 1
    X_dna_seqs_train = list(rand_sequence(num_samples))
    print("Random Sequences (Integer IDs):")
    print(X_dna_seqs_train[:2])
    """
    hint:
        1. You can check X_dna_seqs_train by print, the data is ids which is your training X
        2. You first convert ids back to DNA sequence
        3. Then you run count_cpgs which will yield CGs counts - this will be the labels (Y)
    """
    #step2
    temp = [''.join(list(intseq_to_dnaseq(seq)))  for seq in X_dna_seqs_train]  # use intseq_to_dnaseq here to convert ids back to DNA seqs
    print("Converted DNA Sequences (String Representation):")
    print(temp)
    #step3
    y_dna_seqs = [count_cpgs(seq) for seq in temp]  # use count_cpgs here to generate labels with temp generated in step2
    print("CG Counts (Labels):")
    print(y_dna_seqs)
    return X_dna_seqs_train, y_dna_seqs

train_x, train_y = prepare_data(2048)
test_x, test_y = prepare_data(512)

Random Sequences (Integer IDs):
[[3, 3, 3, 0, 2, 3, 2, 2, 2, 0, 2, 4, 2, 4, 4, 1, 3, 3, 3, 3, 3, 1, 1, 0, 0, 2, 1, 4, 4, 4, 0, 3, 1, 2, 4, 3, 0, 3, 4, 0, 2, 3, 4, 0, 4, 3, 2, 1, 1, 1, 4, 1, 2, 4, 3, 0, 1, 0, 0, 4, 3, 2, 2, 3, 4, 3, 4, 1, 1, 4, 4, 1, 4, 0, 0, 2, 3, 0, 4, 1, 2, 4, 3, 4, 4, 0, 0, 3, 2, 0, 2, 2, 1, 2, 0, 3, 2, 2, 2, 1, 3, 0, 1, 3, 0, 4, 3, 1, 3, 0, 0, 0, 3, 0, 4, 0, 1, 3, 2, 2, 1, 1, 2, 2, 1, 0, 3, 3], [3, 3, 2, 4, 3, 2, 3, 2, 0, 2, 1, 4, 2, 0, 2, 0, 2, 1, 0, 3, 4, 3, 0, 4, 4, 3, 2, 1, 4, 1, 3, 2, 2, 2, 1, 4, 4, 0, 4, 1, 3, 2, 2, 0, 3, 3, 0, 2, 3, 0, 0, 0, 2, 4, 4, 2, 0, 2, 4, 1, 2, 1, 0, 2, 2, 0, 2, 1, 3, 3, 1, 3, 2, 2, 2, 0, 2, 3, 4, 4, 2, 1, 1, 1, 3, 1, 0, 2, 1, 3, 0, 0, 0, 3, 1, 4, 4, 1, 1, 4, 3, 3, 0, 4, 1, 3, 4, 3, 0, 4, 0, 2, 3, 0, 3, 2, 2, 1, 0, 2, 0, 4, 3, 3, 1, 1, 1, 3]]
Converted DNA Sequences (String Representation):
['...']
CG Counts (Labels):
[4, 4, 7, 5, 4, 4, 6, 8, 6, 9, 7, 6, 4, 9, 4, 3, 6, 2, 9, 7, 9, 7, 6, 3, 5, 5, 2, 4, 4, 5, 4, 3, 7, 5, 6, 6, 5, 8, 3, 

In [7]:
print(train_x[:2])
print(train_y[:])

[[3, 3, 3, 0, 2, 3, 2, 2, 2, 0, 2, 4, 2, 4, 4, 1, 3, 3, 3, 3, 3, 1, 1, 0, 0, 2, 1, 4, 4, 4, 0, 3, 1, 2, 4, 3, 0, 3, 4, 0, 2, 3, 4, 0, 4, 3, 2, 1, 1, 1, 4, 1, 2, 4, 3, 0, 1, 0, 0, 4, 3, 2, 2, 3, 4, 3, 4, 1, 1, 4, 4, 1, 4, 0, 0, 2, 3, 0, 4, 1, 2, 4, 3, 4, 4, 0, 0, 3, 2, 0, 2, 2, 1, 2, 0, 3, 2, 2, 2, 1, 3, 0, 1, 3, 0, 4, 3, 1, 3, 0, 0, 0, 3, 0, 4, 0, 1, 3, 2, 2, 1, 1, 2, 2, 1, 0, 3, 3], [3, 3, 2, 4, 3, 2, 3, 2, 0, 2, 1, 4, 2, 0, 2, 0, 2, 1, 0, 3, 4, 3, 0, 4, 4, 3, 2, 1, 4, 1, 3, 2, 2, 2, 1, 4, 4, 0, 4, 1, 3, 2, 2, 0, 3, 3, 0, 2, 3, 0, 0, 0, 2, 4, 4, 2, 0, 2, 4, 1, 2, 1, 0, 2, 2, 0, 2, 1, 3, 3, 1, 3, 2, 2, 2, 0, 2, 3, 4, 4, 2, 1, 1, 1, 3, 1, 0, 2, 1, 3, 0, 0, 0, 3, 1, 4, 4, 1, 1, 4, 3, 3, 0, 4, 1, 3, 4, 3, 0, 4, 0, 2, 3, 0, 3, 2, 2, 1, 0, 2, 0, 4, 3, 3, 1, 1, 1, 3]]
[4, 4, 7, 5, 4, 4, 6, 8, 6, 9, 7, 6, 4, 9, 4, 3, 6, 2, 9, 7, 9, 7, 6, 3, 5, 5, 2, 4, 4, 5, 4, 3, 7, 5, 6, 6, 5, 8, 3, 6, 5, 5, 6, 7, 3, 6, 6, 4, 4, 6, 5, 6, 6, 6, 9, 7, 8, 10, 7, 5, 5, 3, 7, 4, 8, 4, 6, 6, 3, 7, 5, 5, 4, 0, 3, 

In [8]:
# create data loader
class DNASequenceDataset(Dataset):
  def __init__(self, sequences, labels):
    self.sequences = sequences
    self.labels = labels

  def __len__(self):
    return len(self.sequences)

  def __getitem__(self, idx):
    seq = self.sequences[idx]
    #create one-hot tensor (sequence_length, 5)
    one_hot = torch.zeros(len(seq),5)
    for i, s in enumerate(seq):
      one_hot[i][s] = 1
    label = self.labels[idx]
    return one_hot, torch.tensor(self.labels[idx], dtype=torch.float)


In [9]:
batch_size = 128
def create_data_loader(train_x, train_y, test_x, test_y, batch_size):
  train_dataset = DNASequenceDataset(train_x, train_y)
  test_dataset = DNASequenceDataset(test_x, test_y)


  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size)

  return train_loader,  test_loader

train_loader, test_loader = create_data_loader(train_x, train_y, test_x, test_y, batch_size)

In [10]:
for batch in train_loader:
    sequences, labels = batch
    print("Padded Sequences (Batch):", sequences.shape)
    print("Labels (Batch):", labels.shape)
    break

Padded Sequences (Batch): torch.Size([128, 128, 5])
Labels (Batch): torch.Size([128])


In [11]:
# some config
input_size = 5
hidden_size = 128
num_layers = 2
batch_size = 64
learning_rate = 0.001
epoch_num = 20
dropout = 0.2

In [12]:
# Model
class CpGPredictor(torch.nn.Module):
    ''' Simple model that uses a LSTM to count the number of CpGs in a sequence '''
    def __init__(self,input_size, hidden_size, num_layers, dropout):
        super(CpGPredictor, self).__init__()
        # TODO complete model, you are free to add whatever layers you need here
        # We do need a lstm and a classifier layer here but you are free to implement them in your way
        self.lstm = nn.LSTM(
            input_size= input_size,
            hidden_size= hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout
        )
        # Fully connected layer for regression
        self.classifier = nn.Linear(hidden_size, 1)  # Output a single value (CpG count)

    def forward(self, x):
        # TODO complete forward function
        # Pass input through LSTM
        lstm_out, (hidden, cell) = self.lstm(x)

        # Take the final hidden state from the last LSTM layer
        final_hidden = hidden[-1]  # Shape: (batch_size, hidden_size)

        # Pass through the classifier
        logits = self.classifier(final_hidden)  # Shape: (batch_size, 1)
        return logits

In [13]:
model = CpGPredictor(input_size, hidden_size, num_layers, dropout)
criterion = nn.HuberLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [14]:
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
for epoch in range(epoch_num):
    t_loss = 0.0
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    avg_loss = t_loss / len(train_loader)
    scheduler.step(avg_loss)  # Adjust learning rate

    print(f"Epoch [{epoch+1}/{epoch_num}], Loss: {avg_loss}")

Epoch [1/20], Loss: 3.4880290776491165
Epoch [2/20], Loss: 1.2713674902915955
Epoch [3/20], Loss: 1.2378072440624237
Epoch [4/20], Loss: 1.2372396402060986
Epoch [5/20], Loss: 1.2330768220126629
Epoch [6/20], Loss: 1.2319564744830132
Epoch [7/20], Loss: 1.2313704118132591
Epoch [8/20], Loss: 1.2321972772479057
Epoch [9/20], Loss: 1.2329494133591652
Epoch [10/20], Loss: 1.2327448949217796
Epoch [11/20], Loss: 1.2320132851600647
Epoch [12/20], Loss: 1.232699193060398
Epoch [13/20], Loss: 1.232232928276062
Epoch [14/20], Loss: 1.2315799295902252
Epoch [15/20], Loss: 1.2314562574028969
Epoch [16/20], Loss: 1.2321096882224083
Epoch [17/20], Loss: 1.2313604094088078
Epoch [18/20], Loss: 1.2319929152727127
Epoch [19/20], Loss: 1.2323862463235855
Epoch [20/20], Loss: 1.2314140647649765


In [15]:
# some config
input_size = 5
hidden_size = 128
num_layers1 = 3
batch_size = 64
learning_rate1 = 0.0005
epoch_num = 10
dropout = 0.2

model = CpGPredictor(input_size, hidden_size, num_layers1, dropout)
criterion = nn.HuberLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate1)

In [16]:
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
for epoch in range(epoch_num):
    t_loss = 0.0
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    avg_loss = t_loss / len(train_loader)
    scheduler.step(avg_loss)  # Adjust learning rate

    print(f"Epoch [{epoch+1}/{epoch_num}], Loss: {avg_loss}")

Epoch [1/10], Loss: 4.24967160820961
Epoch [2/10], Loss: 1.6710286065936089
Epoch [3/10], Loss: 1.2611631080508232
Epoch [4/10], Loss: 1.2393177151679993
Epoch [5/10], Loss: 1.2305526360869408
Epoch [6/10], Loss: 1.2335634604096413
Epoch [7/10], Loss: 1.234772838652134
Epoch [8/10], Loss: 1.2328040152788162
Epoch [9/10], Loss: 1.231971189379692
Epoch [10/10], Loss: 1.231190174818039


In [24]:
# Define combinations of learning rates
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate_combinations = [0.001, 0.0005, 0.0001, 0.01]
best_validation_loss = float('inf')
best_lr = None

# Loop through each learning rate combination
for lr in learning_rate_combinations:
    print(f"Training with learning rate: {lr}")

    # Model, criterion, optimizer, and scheduler setup
    model = CpGPredictor(input_size, hidden_size, num_layers, dropout).to(device)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

    # Training loop
    for epoch in range(epoch_num):
        model.train()
        t_loss = 0.0
        v_loss = 0.0  # Initialize validation loss

        # Split data into training and validation sets (80-20 split)
        num_training_samples = int(len(train_loader.dataset) * 0.8)
        train_subset, validation_subset = torch.utils.data.random_split(
            train_loader.dataset, [num_training_samples, len(train_loader.dataset) - num_training_samples]
        )

        train_subloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size)
        validation_subloader = torch.utils.data.DataLoader(validation_subset, batch_size=batch_size)

        # Training phase
        for batch_x, batch_y in train_subloader:  # Use train_subloader here
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs.squeeze(1), batch_y)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        # Validation phase
        model.eval()
        with torch.no_grad():
            for batch_x, batch_y in validation_subloader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                loss = criterion(outputs.squeeze(1), batch_y)
                v_loss += loss.item()

        avg_loss = t_loss / len(train_subloader)  # Use len(train_subloader)
        avg_validation_loss = v_loss / len(validation_subloader)

        # Scheduler step
        scheduler.step(avg_validation_loss)

        # Print epoch results
        print(f"Epoch [{epoch+1}/{epoch_num}], Loss: {avg_loss:.4f}, Validation Loss: {avg_validation_loss:.4f}")

    # Check if the current learning rate gives the best validation loss
    if avg_validation_loss < best_validation_loss:
        best_validation_loss = avg_validation_loss
        best_lr = lr

print(f"Best learning rate: {best_lr}, Best validation loss: {best_validation_loss}")

Training with learning rate: 0.001
Epoch [1/10], Loss: 2.7040, Validation Loss: 1.2232
Epoch [2/10], Loss: 1.2442, Validation Loss: 1.1980
Epoch [3/10], Loss: 1.2219, Validation Loss: 1.2986
Epoch [4/10], Loss: 1.2117, Validation Loss: 1.2908
Epoch [5/10], Loss: 1.2489, Validation Loss: 1.1578
Epoch [6/10], Loss: 1.2362, Validation Loss: 1.1724
Epoch [7/10], Loss: 1.2153, Validation Loss: 1.3464
Epoch [8/10], Loss: 1.2229, Validation Loss: 1.2754
Epoch [9/10], Loss: 1.2109, Validation Loss: 1.3459
Epoch [10/10], Loss: 1.2406, Validation Loss: 1.2406
Training with learning rate: 0.0005
Epoch [1/10], Loss: 3.7199, Validation Loss: 1.3170
Epoch [2/10], Loss: 1.2435, Validation Loss: 1.2844
Epoch [3/10], Loss: 1.2485, Validation Loss: 1.1562
Epoch [4/10], Loss: 1.2442, Validation Loss: 1.2058
Epoch [5/10], Loss: 1.2223, Validation Loss: 1.2989
Epoch [6/10], Loss: 1.2348, Validation Loss: 1.1978
Epoch [7/10], Loss: 1.2343, Validation Loss: 1.2497
Epoch [8/10], Loss: 1.2418, Validation Loss:

In [25]:
model = CpGPredictor(input_size, hidden_size=128, num_layers=3, dropout=0.3)
criterion = nn.HuberLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

In [26]:
for epoch in range(epoch_num):
    t_loss = 0.0
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    avg_loss = t_loss / len(train_loader)
    scheduler.step(avg_loss)  # Adjust learning rate

    print(f"Epoch [{epoch+1}/{epoch_num}], Loss: {avg_loss}")

Epoch [1/10], Loss: 4.12559175491333
Epoch [2/10], Loss: 1.5631696432828903
Epoch [3/10], Loss: 1.2498573809862137
Epoch [4/10], Loss: 1.235059104859829
Epoch [5/10], Loss: 1.2324042469263077
Epoch [6/10], Loss: 1.2341439947485924
Epoch [7/10], Loss: 1.231564998626709
Epoch [8/10], Loss: 1.2327173501253128
Epoch [9/10], Loss: 1.2322163134813309
Epoch [10/10], Loss: 1.2319052517414093


In [27]:
model = CpGPredictor(input_size, hidden_size=128, num_layers=4, dropout=0.3)
criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

In [28]:
for epoch in range(epoch_num):
    t_loss = 0.0
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    avg_loss = t_loss / len(train_loader)
    scheduler.step(avg_loss)  # Adjust learning rate

    print(f"Epoch [{epoch+1}/{epoch_num}], Loss: {avg_loss}")

Epoch [1/10], Loss: 2.309852361679077
Epoch [2/10], Loss: 0.9767917133867741
Epoch [3/10], Loss: 0.7331732586026192
Epoch [4/10], Loss: 0.718084029853344
Epoch [5/10], Loss: 0.7174734100699425
Epoch [6/10], Loss: 0.7172675877809525
Epoch [7/10], Loss: 0.7168965041637421
Epoch [8/10], Loss: 0.7174390517175198
Epoch [9/10], Loss: 0.7175025306642056
Epoch [10/10], Loss: 0.7167525850236416


In [30]:
model = CpGPredictor(input_size, hidden_size=128, num_layers=4, dropout=0.3)
criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

best_val_loss = float('inf')

for epoch in range(epoch_num):
    t_loss = 0.0
    model.train()
    for batch_x, batch_y in train_subloader:  # Make sure train_subloader is defined
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    avg_loss = t_loss / len(train_subloader)

    # Validation loop
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in validation_subloader:  # Make sure validation_subloader is defined
            outputs = model(batch_x)
            loss = criterion(outputs.squeeze(1), batch_y)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(validation_subloader)

    scheduler.step(avg_val_loss)  # Use validation loss for scheduling

    print(f"Epoch [{epoch+1}/{epoch_num}], Train Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # Save the model if validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')

Epoch [1/10], Train Loss: 1.8543, Val Loss: 0.7449
Epoch [2/10], Train Loss: 0.7447, Val Loss: 0.6845
Epoch [3/10], Train Loss: 0.7252, Val Loss: 0.6781
Epoch [4/10], Train Loss: 0.7261, Val Loss: 0.6790
Epoch [5/10], Train Loss: 0.7258, Val Loss: 0.6786
Epoch [6/10], Train Loss: 0.7258, Val Loss: 0.6787
Epoch [7/10], Train Loss: 0.7256, Val Loss: 0.6785
Epoch [8/10], Train Loss: 0.7255, Val Loss: 0.6794
Epoch [9/10], Train Loss: 0.7247, Val Loss: 0.6793
Epoch [10/10], Train Loss: 0.7251, Val Loss: 0.6793


In [32]:
# Evaluate on test set
model = model.to(device)
model.eval()
test_loss = 0.0
all_predictions = []
all_targets = []
with torch.no_grad():
    for batch_x, batch_y in test_loader:  # Assuming you have a test_loader
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        test_loss += loss.item()

        # Assuming outputs and batch_y are tensors of CpG counts
        predictions = outputs.squeeze(1)
        targets = batch_y

        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
print(f"Test Loss: {avg_test_loss:.4f}")

Test Loss: 0.7093


In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

threshold = 2
predicted_labels = [1 if abs(pred - target) <= threshold else 0 for pred, target in zip(all_predictions, all_targets)]
true_labels = [1 if abs(pred - target) <= threshold else 0 for pred, target in zip(all_predictions, all_targets)]

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-score: 1.0000
Confusion Matrix:
[[174   0]
 [  0 338]]


In [44]:
# Function to highlight 'CG' in the sequence (for terminal-based visualization)
def highlight_cg(sequence):
    return sequence.replace('CG', '\033[1;31mCG\033[0m')  # Red color for 'CG' in terminal

# Generate a random DNA sequence (example)
random_sequence = ''.join(random.choice('NACGT') for _ in range(128))

# Highlight 'CG' in the sequence for better visibility
highlighted_sequence = highlight_cg(random_sequence)

# Count the actual number of 'CG' subsequences
actual_cpg_count = random_sequence.count('CG')

# Convert the sequence to numerical representation
int_sequence = list(map(dna2int.get, random_sequence))
int_sequence_tensor = torch.tensor(int_sequence, dtype=torch.long).unsqueeze(0)

# Create a one-hot tensor of the input sequence
one_hot_tensor = torch.zeros(int_sequence_tensor.shape[0], int_sequence_tensor.shape[1], len(dna2int))
one_hot_tensor = one_hot_tensor.scatter(2, int_sequence_tensor.unsqueeze(2), 1)

# Make the prediction
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device)
with torch.no_grad():
    prediction = model(one_hot_tensor.float().to(device))

# Process the prediction
predicted_cpg_count = torch.round(prediction).item()

# Print the results with highlighted 'CG' subsequences
print(f"Random Sequence: {highlighted_sequence}")
print(f"Actual CpG Count: {actual_cpg_count}")
print(f"Predicted CpG Count: {predicted_cpg_count}")

Random Sequence: GCNTAGTNATGNNGTTCATCGTGANNGCGCCCGACACCATANNANGGGCATACGCNTNCAATTAGACCACTCNGCCCAACGACCCNTGGTTNCNTAATTNCNCCGAATTCATTATGGCCGCGCNTTGC
Actual CpG Count: 8
Predicted CpG Count: 5.0


# Part 2: what if the DNA sequences are not the same length

In [71]:
# hint we will need following imports
import random
from functools import partial
from typing import Sequence
import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

In [72]:
# DO NOT CHANGE HERE
random.seed(13)

# Use this for getting x label
def rand_sequence_var_len(n_seqs: int, lb: int=16, ub: int=128) -> Sequence[int]:
    for i in range(n_seqs):
        seq_len = random.randint(lb, ub)
        yield [random.randint(1, 5) for _ in range(seq_len)]


# Use this for getting y label
def count_cpgs(seq: str) -> int:
    cgs = 0
    for i in range(0, len(seq) - 1):
        dimer = seq[i:i+2]
        # note that seq is a string, not a list
        if dimer == "CG":
            cgs += 1
    return cgs


# Alphabet helpers
alphabet = 'NACGT'
dna2int = {a: i for a, i in zip(alphabet, range(1, 6))}
int2dna = {i: a for a, i in zip(alphabet, range(1, 6))}
dna2int.update({"pad": 0})
int2dna.update({0: "<pad>"})

intseq_to_dnaseq = partial(map, int2dna.get)
dnaseq_to_intseq = partial(map, dna2int.get)

In [73]:
# TODO complete the task based on the change
def prepare_data(num_samples=100, min_len=16, max_len=128):
    # prepared the training and test data
    # you need to call rand_sequence and count_cpgs here to create the dataset
    #step 1
    X_int_seqs_train = list(rand_sequence_var_len(num_samples, min_len, max_len))
    #step 2: Convert DNA sequences to integer sequences using dna2int.get()
    temp = [''.join(list(intseq_to_dnaseq(seq)))  for seq in X_int_seqs_train]
    #step3: Calculate CpG counts, handling potential None values
    y_dna_seqs = [count_cpgs(seq) for seq in temp]

    return X_int_seqs_train, y_dna_seqs

min_len, max_len = 64, 128
train_x, train_y = prepare_data(2048, min_len, max_len)
test_x, test_y = prepare_data(512, min_len, max_len)

In [101]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, lists, labels) -> None:
        self.lists = lists
        self.labels = labels
        self.lengths = [len(lst) for lst in lists]

    def __getitem__(self, index):
        return torch.LongTensor(self.lists[index]), self.labels[index], self.lengths[index]

    def __len__(self):
        return len(self.lists)


class PadSequence:
    def __call__(self, batch):
        try:
            # Get sequences and labels from the batch
            sequences, labels = zip(*batch)

            # Get lengths before padding
            lengths = [len(seq) for seq in sequences]

            # Convert sequences to tensors and pad them
            sequences = [torch.LongTensor(seq) for seq in sequences]
            padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)

            # Convert lengths and labels to tensors
            lengths = torch.LongTensor(lengths)
            labels = torch.FloatTensor(labels)

            return padded_sequences, lengths, labels

        except Exception as e:
            print("Batch content:", batch)
            print("Batch length:", len(batch))
            if len(batch) > 0:
                print("First item type:", type(batch[0]))
                print("First item content:", batch[0])
            raise e

In [103]:
# Create an instance of the PadSequence class
pad_sequence = PadSequence()

# Create the DataLoader with the collate_fn argument
batch_size = 64
train_dataset = MyDataset(train_x, train_y)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=pad_sequence,
    shuffle=True  # Add shuffling for training
)

In [104]:
class CpGPredictor(torch.nn.Module):
       def __init__(self, input_size, hidden_size, num_layers, dropout):
           super(CpGPredictor, self).__init__()
           self.lstm = nn.LSTM(
               input_size=input_size,
               hidden_size=hidden_size,
               num_layers=num_layers,
               batch_first=True,
               dropout=dropout
           )
           self.classifier = nn.Linear(hidden_size, 1)

       def forward(self, x, lengths):  # Add lengths argument
           # Pack the padded sequence
           packed_sequence = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

           # Pass through LSTM
           lstm_out, (hidden, cell) = self.lstm(packed_sequence)

           # Unpack the padded sequence
           output, _ = pad_packed_sequence(lstm_out, batch_first=True)

           # Take the final hidden state
           final_hidden = hidden[-1]

           # Pass through the classifier
           logits = self.classifier(final_hidden)
           return logits

In [99]:
# Model initialization
input_size = 5  # Size of your vocabulary (NACGT + padding)
hidden_size = 64
num_layers = 2
dropout = 0.1

# Initialize model and move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CpGPredictor(input_size, hidden_size, num_layers, dropout).to(device)

# Initialize optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch_x, lengths, batch_y in train_loader:
        # Move to device
        batch_x = batch_x.to(device)
        lengths = lengths.to(device)
        batch_y = batch_y.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_x, lengths)

        # Compute loss
        loss = criterion(outputs.squeeze(1), batch_y)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

In [ ]:
model = CpGPredictor(input_size, hidden_size=128, num_layers=4, dropout=0.3)
criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

for epoch in range(epoch_num):
    t_loss = 0.0
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.squeeze(1), batch_y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    avg_loss = t_loss / len(train_loader)
    scheduler.step(avg_loss)  # Adjust learning rate

    print(f"Epoch [{epoch+1}/{epoch_num}], Loss: {avg_loss}")
